In [157]:
import bagpy
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from bagpy import bagreader

In [158]:
# b = bagreader("/home/repos/Neo/data/raw_bagfiles/translate/single/mustard_bottle_ros_topics.bag")
b = bagreader("/home/repos/Neo/data/raw_bagfiles/translate/single/tomato_soup_can_ros_topics.bag")
# b = bagreader("/home/repos/Neo/data/raw_bagfiles/translate_and_rotate/single/mustard_bottle_ros_topics.bag")

for topic in b.topics:
    if 'gripper' in topic:
        print(topic)

[INFO]  Data folder /home/repos/Neo/data/raw_bagfiles/translate/single/tomato_soup_can_ros_topics already exists. Not creating.
/trolley/panda/gripper_controller/grasp/goal
/trolley/panda/gripper_controller/grasp/result
/trolley/panda/gripper_controller/grasp/status
/trolley/panda/gripper_controller/gripper_action/status
/trolley/panda/gripper_controller/homing/status
/trolley/panda/gripper_controller/joint_states
/trolley/panda/gripper_controller/move/goal
/trolley/panda/gripper_controller/move/result
/trolley/panda/gripper_controller/move/status
/trolley/panda/gripper_controller/stop/status


In [159]:
arm_topics = "/trolley/panda/armer/state"
data = b.message_by_topic(arm_topics)
df = pd.read_csv(data)

print("############# Arm topics #############")
for column in df.columns:
    print(column)

gripper_topic = "/trolley/panda/gripper_controller/joint_states"
data = b.message_by_topic(gripper_topic)
df_gripper = pd.read_csv(data)

print("############# Grippper topics #############")
for column in df_gripper.columns:
    print(column)

############# Arm topics #############
Time
ee_pose.header.seq
ee_pose.header.stamp.secs
ee_pose.header.stamp.nsecs
ee_pose.header.frame_id
ee_pose.pose.position.x
ee_pose.pose.position.y
ee_pose.pose.position.z
ee_pose.pose.orientation.x
ee_pose.pose.orientation.y
ee_pose.pose.orientation.z
ee_pose.pose.orientation.w
ee_velocity.header.seq
ee_velocity.header.stamp.secs
ee_velocity.header.stamp.nsecs
ee_velocity.header.frame_id
ee_velocity.twist.linear.x
ee_velocity.twist.linear.y
ee_velocity.twist.linear.z
ee_velocity.twist.angular.x
ee_velocity.twist.angular.y
ee_velocity.twist.angular.z
ee_wrench.header.seq
ee_wrench.header.stamp.secs
ee_wrench.header.stamp.nsecs
ee_wrench.header.frame_id
ee_wrench.wrench.force.x
ee_wrench.wrench.force.y
ee_wrench.wrench.force.z
ee_wrench.wrench.torque.x
ee_wrench.wrench.torque.y
ee_wrench.wrench.torque.z
joint_poses_0
joint_poses_1
joint_poses_2
joint_poses_3
joint_poses_4
joint_poses_5
joint_poses_6
joint_velocities_0
joint_velocities_1
joint_velo

In [ ]:
column_names = [f"ee_pose.pose.position.{x}" for x in ['x', 'y', 'z']]
column_names.extend([f"ee_pose.pose.orientation.{x}" for x in ['x', 'y', 'z', 'w']])
df[column_names] = df[column_names].round(3)
column_names.extend(['Time'])
df = df[column_names]

column_names = [f"position_{x}" for x in range(2)]
df_gripper[column_names] = df_gripper[column_names].round(3)
column_names.extend(['Time'])
df_gripper = df_gripper[column_names]

     position_0  position_1          Time
0           0.0        0.04  1.726553e+09
1           0.0        0.04  1.726553e+09
2           0.0        0.04  1.726553e+09
3           0.0        0.04  1.726553e+09
4           0.0        0.04  1.726553e+09
..          ...         ...           ...
545         0.0        0.04  1.726553e+09
546         0.0        0.04  1.726553e+09
547         0.0        0.04  1.726553e+09
548         0.0        0.04  1.726553e+09
549         0.0        0.04  1.726553e+09

[550 rows x 3 columns]


In [161]:
print(df)

       ee_pose.pose.position.x  ee_pose.pose.position.y  \
0                       -0.340                   -0.287   
1                       -0.340                   -0.287   
2                       -0.340                   -0.287   
3                       -0.340                   -0.287   
4                       -0.340                   -0.287   
...                        ...                      ...   
21099                   -0.337                   -0.299   
21100                   -0.337                   -0.299   
21101                   -0.337                   -0.299   
21102                   -0.337                   -0.299   
21103                   -0.337                   -0.299   

       ee_pose.pose.position.z  ee_pose.pose.orientation.x  \
0                        0.962                      -0.411   
1                        0.962                      -0.411   
2                        0.962                      -0.411   
3                        0.962             

In [162]:
df_gripper['position_0'] = np.abs(df_gripper['position_0'] - df_gripper['position_0'][0]).round(3)
gripper_closed_data = df_gripper[df_gripper['position_0'] == df_gripper[df_gripper['position_0'] > 0]['position_0'].value_counts().idxmax()]['Time']


In [163]:
time_closing = gripper_closed_data.iloc[0]
time_opening = gripper_closed_data.iloc[-1]

print(time_closing, time_opening)

arm_movement = df[np.logical_and(df['Time'] >= time_closing, df['Time'] <= time_opening)]
movement_start = arm_movement.iloc[0].round(3)
movement_end = arm_movement.iloc[-1].round(3)

print(movement_start)
print(movement_end)

1726553282.1235096 1726553301.7561555
ee_pose.pose.position.x      -3.560000e-01
ee_pose.pose.position.y      -6.740000e-01
ee_pose.pose.position.z       6.530000e-01
ee_pose.pose.orientation.x   -3.820000e-01
ee_pose.pose.orientation.y    9.240000e-01
ee_pose.pose.orientation.z    1.000000e-03
ee_pose.pose.orientation.w    0.000000e+00
Time                          1.726553e+09
Name: 6062, dtype: float64
ee_pose.pose.position.x      -2.030000e-01
ee_pose.pose.position.y      -4.670000e-01
ee_pose.pose.position.z       6.540000e-01
ee_pose.pose.orientation.x   -3.830000e-01
ee_pose.pose.orientation.y    9.240000e-01
ee_pose.pose.orientation.z   -3.000000e-03
ee_pose.pose.orientation.w   -1.000000e-03
Time                          1.726553e+09
Name: 15855, dtype: float64


In [164]:
from scipy.spatial.transform import Rotation

total_travel = (np.round(np.abs(movement_end - movement_start)[['ee_pose.pose.position.x', 'ee_pose.pose.position.y', 'ee_pose.pose.position.z']], 3)).tolist()
quat_start = Rotation.from_quat(movement_start[['ee_pose.pose.orientation.x', 'ee_pose.pose.orientation.y', 'ee_pose.pose.orientation.z', 'ee_pose.pose.orientation.w']].to_numpy())
quat_end = Rotation.from_quat(movement_end[['ee_pose.pose.orientation.x', 'ee_pose.pose.orientation.y', 'ee_pose.pose.orientation.z', 'ee_pose.pose.orientation.w']].to_numpy())

total_rotation = quat_start * quat_end.inv()
total_rotation = np.round(total_rotation.as_euler('xyz', degrees=True), 3)

print(total_travel)
print(total_rotation)

[0.153, 0.207, 0.001]
[ 0.467  0.07  -0.106]
